<a href="https://colab.research.google.com/github/RadSpiral/Accuracy-of-the-k--fold-Cross-Validation-in-High-Dimensions/blob/main/neural_net_kcv_sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Simulation: 2-layer MLP on synthetic data.
Generate synthetic data, train with K-fold CV, compare to test MSE.

Colab version with local checkpointing.
Results are saved to ./results/checkpoint.pkl during the run.
Note: Results will be lost if the Colab session disconnects.
"""

import os
import pickle

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIG (set here)
# ============================================================================
hidden_size = 5
gamma = 0.5
n_repetitions = 50
d_values = [100, 200, 400, 800]
fixed_initialization = False  # True = same init for all folds/reps; False = random each time

learning_rate = 0.05
batch_size = 16
total_steps = 100      # FIXED total GD steps per fit; epochs derived from data size
tiny_noise_scale = 1e-5
noise_std = 0.1          # std of additive noise on outputs

def epochs_for(n):
    """Epochs implied by the fixed step budget for a fit on n samples."""
    steps_per_epoch = int(np.ceil(n / batch_size))
    return max(1, int(np.ceil(total_steps / steps_per_epoch)))

seed = 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================================
# CHECKPOINTING (local storage)
# ============================================================================
RESULTS_DIR = './results'
os.makedirs(RESULTS_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(RESULTS_DIR, 'checkpoint.pkl')

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, 'rb') as f:
            return pickle.load(f)
    return {}

def save_checkpoint(done):
    tmp = CHECKPOINT_PATH + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(done, f)
    os.replace(tmp, CHECKPOINT_PATH)  # atomic, no corrupt file if killed mid-write

# done maps (rule_name, d, rep) -> dict with keys
#   'test_MSE', 'KCV', and for rep 0 also 'train_loss_final'
done = load_checkpoint()
if done:
    print(f"Resuming: {len(done)} completed reps found in {CHECKPOINT_PATH}")

def seed_for(rule_idx, d, rep):
    """Deterministic per-rep seeding. Results do not depend on whether the
    run was interrupted, and reps can in principle run in any order."""
    ss = np.random.SeedSequence([seed, rule_idx, d, rep])
    s = int(ss.generate_state(1)[0])
    np.random.seed(s % (2**32))
    torch.manual_seed(s)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def weight_scale_w1(d):
    """Initialization scale for first layer: 1/sqrt(d)"""
    return 1.0 / np.sqrt(d)

def weight_scale_w2():
    """Initialization scale for second layer: 1/sqrt(hidden_size)"""
    return 1.0 / np.sqrt(hidden_size)

class TwoLayerMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = hidden_size
        self.fc1 = nn.Linear(input_dim, hidden_dim, bias=False)
        self.fc2 = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, x):
        h = torch.relu(self.fc1(x))
        y = self.fc2(h)
        return y

def compute_p(d):
    """Parameter count, computed directly from the model so it always
    matches the architecture regardless of hidden_size."""
    return sum(q.numel() for q in TwoLayerMLP(d).parameters() if q.requires_grad)

def init_weights(model, d, fixed=False):
    """Initialize weights (no biases). If fixed=False, use fresh random init.
    X is unscaled, so W1 is 1/sqrt(d); W2 is 1/sqrt(h)."""
    if not fixed:
        # Random init with proper scaling
        with torch.no_grad():
            model.fc1.weight.normal_(0, weight_scale_w1(d))      # 1/sqrt(d)
            model.fc2.weight.normal_(0, weight_scale_w2())       # 1/sqrt(h)
    else:
        # Fixed init: same for all fits at this dimension
        with torch.no_grad():
            model.fc1.weight.normal_(0, tiny_noise_scale)
            model.fc2.weight.normal_(0, tiny_noise_scale)

def forward_pass(X, W1, W2):
    """Compute output of true network: h = ReLU(X @ W1), y = h @ W2
    X is unscaled standard normal."""
    h = np.maximum(X @ W1, 0)  # ReLU
    y = h @ W2
    return y

def train_model(model, X_train, y_train, lr=0.005, batch_size=16, epochs=100, verbose=False):
    """Train model on data using GD. Return trained model and loss history."""
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device=device).reshape(-1, 1)

    n = X_train.shape[0]
    loss_history = []
    print_every = max(1, epochs // 3)

    for epoch in range(epochs):
        # Shuffle
        perm = np.random.permutation(n)
        X_sh = X_train_tensor[perm]
        y_sh = y_train_tensor[perm]

        epoch_loss = 0.0
        for i in range(0, n, batch_size):
            X_batch = X_sh[i:i+batch_size]
            y_batch = y_sh[i:i+batch_size]

            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(X_batch)

        epoch_loss /= n
        loss_history.append(epoch_loss)

        if verbose and (epoch + 1) % print_every == 0:
            print(f"      Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.6f}")

    return model, loss_history

def evaluate_model(model, X, y):
    """Evaluate MSE on data."""
    model.eval()
    X_tensor = torch.tensor(X, dtype=torch.float32, device=device)
    y_tensor = torch.tensor(y, dtype=torch.float32, device=device).reshape(-1, 1)

    with torch.no_grad():
        pred = model(X_tensor)
        mse = ((pred - y_tensor) ** 2).mean().item()

    return mse

def test_mse_streaming(model, W1_true, W2_true,
                       n_test, d, noise_std, chunk_size=10000):
    """Compute out-of-sample MSE without storing the test set.
    Generates test data chunk by chunk, evaluates, accumulates."""
    model.eval()
    total_sq_err = 0.0
    remaining = n_test
    with torch.no_grad():
        while remaining > 0:
            m = min(chunk_size, remaining)
            X_chunk = np.random.normal(0, 1.0, size=(m, d)).astype(np.float32)
            y_chunk = forward_pass(X_chunk, W1_true, W2_true).flatten() \
                      + np.random.normal(0, noise_std, size=m)
            X_t = torch.tensor(X_chunk, dtype=torch.float32, device=device)
            y_t = torch.tensor(y_chunk, dtype=torch.float32, device=device).reshape(-1, 1)
            pred = model(X_t)
            total_sq_err += ((pred - y_t) ** 2).sum().item()
            remaining -= m
    return total_sq_err / n_test

# ============================================================================
# MAIN SIMULATION
# ============================================================================

print("=" * 70)
print("SIMULATION: 2-Layer MLP with K-Fold CV")
print("=" * 70)
print(f"Config: gamma={gamma}, n_reps={n_repetitions}, hidden={hidden_size}")
print(f"Optimization: lr={learning_rate}, batch_size={batch_size}, total_steps={total_steps}")
print(f"Fixed init: {fixed_initialization}")
print(f"Device: {device}")
print()

K_RULES = [
    ("K=4",      lambda n: 4),
    ("K=n^0.25", lambda n: max(2, int(np.ceil(n ** 0.25)))),
    ("K=n^0.5",  lambda n: max(2, int(np.ceil(n ** 0.5)))),
    ("K=n^0.75", lambda n: max(2, int(np.ceil(n ** 0.75)))),
]

from scipy import stats as sps

all_rate_points = {}   # rule_name -> list of (p, |KCV - test_MSE|)

def true_network(d):
    """True network for dimension d, identical across settings
    (dedicated RNG seeded by d). X is unscaled, so W1 is 1/sqrt(d); W2 is 1/sqrt(h)."""
    rng = np.random.RandomState(seed + d)
    W1 = rng.normal(0, 1.0/np.sqrt(d), size=(d, hidden_size))
    W2 = rng.normal(0, 1.0 / np.sqrt(hidden_size), size=(hidden_size, 1))
    return W1, W2

for rule_idx, (rule_name, K_of_n) in enumerate(K_RULES):
    print("\n" + "#" * 70)
    print(f"# SETTING: {rule_name}")
    print("#" * 70)

    rate_points = []   # (p, |KCV - test_MSE|), one per rep per dimension

    for d in d_values:
        p = compute_p(d)
        n_train = int(gamma * p)
        n_test = 10 * n_train
        K = K_of_n(n_train)

        W1_true, W2_true = true_network(d)

        print(f"\n  --- d={d}, p={p}, n_train={n_train}, n/p={n_train/p:.3f}, K={K} ---")

        n_skipped = sum(1 for rep in range(n_repetitions) if (rule_name, d, rep) in done)
        if n_skipped:
            print(f"    (skipping {n_skipped} already-completed reps)")

        for rep in range(n_repetitions):
            key = (rule_name, d, rep)
            if key in done:
                continue

            # Deterministic seeding for this rep. Everything below
            # (data, fold splits via shuffling state, inits, batching)
            # flows from this seed.
            seed_for(rule_idx, d, rep)

            # Generate training data (with additive noise on outputs)
            # Features X unscaled standard normal
            X_train = np.random.normal(0, 1.0, size=(n_train, d))
            y_train = forward_pass(X_train, W1_true, W2_true).flatten() \
                      + np.random.normal(0, noise_std, size=n_train)

            # Test data is NOT stored; generated in chunks at evaluation time

            # ===== K-FOLD CV =====
            kfold = KFold(n_splits=K, shuffle=True, random_state=rep)
            fold_losses = []

            for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train)):
                X_train_fold = X_train[train_idx]
                y_train_fold = y_train[train_idx]
                X_val_fold = X_train[val_idx]
                y_val_fold = y_train[val_idx]

                model = TwoLayerMLP(d, hidden_dim=hidden_size)
                init_weights(model, d, fixed=fixed_initialization)

                model, _ = train_model(model, X_train_fold, y_train_fold,
                                       lr=learning_rate, batch_size=batch_size,
                                       epochs=epochs_for(len(X_train_fold)),
                                       verbose=False)

                val_mse = evaluate_model(model, X_val_fold, y_val_fold)
                fold_losses.append(val_mse)

            KCV_estimate = np.mean(fold_losses)

            # ===== TEST MSE (train on full data) =====
            verbose_flag = (rep == 0)
            if verbose_flag:
                print(f"    Convergence (rep 1, full-data fit, n={n_train}):")

            model = TwoLayerMLP(d, hidden_dim=hidden_size)
            init_weights(model, d, fixed=fixed_initialization)
            model, loss_hist = train_model(model, X_train, y_train,
                                           lr=learning_rate, batch_size=batch_size,
                                           epochs=epochs_for(n_train),
                                           verbose=verbose_flag)

            test_MSE = test_mse_streaming(model, W1_true, W2_true,
                                          n_test, d, noise_std)

            entry = {'test_MSE': test_MSE, 'KCV': KCV_estimate}
            if rep == 0:
                entry['train_loss_final'] = loss_hist[-1]

            done[key] = entry
            save_checkpoint(done)

        # ===== SUMMARY for this dimension (from checkpoint, works on resume) =====
        results = np.array([[done[(rule_name, d, rep)]['test_MSE'],
                             done[(rule_name, d, rep)]['KCV']]
                            for rep in range(n_repetitions)])
        test_MSE_vals = results[:, 0]
        KCV_vals = results[:, 1]
        diffs = test_MSE_vals - KCV_vals

        rep0 = done[(rule_name, d, 0)]
        if 'train_loss_final' in rep0:
            print(f"    Train loss (final, rep 1): {rep0['train_loss_final']:.6f}")
        print(f"    Test MSE (mean ± std): {np.mean(test_MSE_vals):.6f} ± {np.std(test_MSE_vals):.6f}")
        print(f"    KCV (mean ± std): {np.mean(KCV_vals):.6f} ± {np.std(KCV_vals):.6f}")
        print(f"    Estimation error (Test - KCV): mean={np.mean(diffs):.6f}, std={np.std(diffs):.6f}")
        print(f"    Mean |KCV - Test MSE|: {np.mean(np.abs(diffs)):.6f}")

        for diff_val in np.abs(diffs):
            rate_points.append((p, diff_val))

    all_rate_points[rule_name] = rate_points

    # ===== RATE REGRESSION for this setting =====
    ps = np.array([q[0] for q in rate_points], dtype=float)
    abs_diffs = np.array([q[1] for q in rate_points], dtype=float)
    valid = abs_diffs > 0

    print(f"\n  RATE ESTIMATION [{rule_name}]: 2*log|KCV - test MSE| ~ a + b*log p")
    if valid.sum() < 3:
        print("    not enough valid points for regression")
        continue

    log_p = np.log(ps[valid])
    y = 2.0 * np.log(abs_diffs[valid])

    reg = sps.linregress(log_p, y)
    slope, intercept, stderr = reg.slope, reg.intercept, reg.stderr
    dof = valid.sum() - 2
    t_crit = sps.t.ppf(0.975, dof)

    print(f"    points: {valid.sum()}")
    print(f"    slope b = {slope:.4f}, 95% CI [{slope - t_crit*stderr:.4f}, {slope + t_crit*stderr:.4f}]")
    print(f"    intercept a = {intercept:.4f}, R^2 = {reg.rvalue**2:.4f}")

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

# ============================================================================
# FIGURES (saved locally)
# ============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Save configuration metadata
config_text = f"""K-Fold CV Simulation Configuration
=====================================
hidden_size: {hidden_size}
gamma: {gamma}
n_repetitions: {n_repetitions}
d_values: {d_values}
learning_rate: {learning_rate}
batch_size: {batch_size}
total_steps: {total_steps}
noise_std: {noise_std}
seed: {seed}
"""
config_path = os.path.join(RESULTS_DIR, 'config.txt')
with open(config_path, 'w') as f:
    f.write(config_text)
print(f"Saved config: {config_path}")

for rule_name, pts in all_rate_points.items():        # one figure per K setting
    ps = np.array([q[0] for q in pts], dtype=float)
    ad = np.array([q[1] for q in pts], dtype=float)
    keep = ad > 0
    ps, ad = ps[keep], ad[keep]
    if len(ps) < 3:
        continue

    y = 2.0 * np.log(ad)
    reg = sps.linregress(np.log(ps), y)
    t_crit = sps.t.ppf(0.975, len(ps) - 2)
    ci_lo = reg.slope - t_crit * reg.stderr
    ci_hi = reg.slope + t_crit * reg.stderr

    unique_ps = np.sort(np.unique(ps))
    positions = np.log(unique_ps)
    width = 0.5 * np.min(np.diff(positions)) if len(positions) > 1 else 0.4

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.boxplot([y[ps == u] for u in unique_ps], positions=positions, widths=width)
    x_line = np.linspace(positions[0], positions[-1], 100)
    ax.plot(x_line, reg.intercept + reg.slope * x_line, linewidth=2.5,
            label=f"Slope: {reg.slope:.4f}, 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    ax.set_xticks(positions)
    ax.set_xticklabels([str(int(u)) for u in unique_ps])
    ax.set_xlabel("p")
    ax.set_ylabel("2 log absolute difference")
    ax.set_title(f"CV Error vs Model Complexity ({rule_name}, steps={total_steps})")
    ax.legend(loc="upper right")
    plt.tight_layout()

    safe_name = rule_name.replace('^', '').replace('=', '').replace('.', '')
    out_path = os.path.join(RESULTS_DIR, f"rate_{safe_name}.png")
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"Saved figure: {out_path}")

SIMULATION: 2-Layer MLP with K-Fold CV
Config: gamma=0.5, n_reps=50, hidden=5
Optimization: lr=0.05, batch_size=16, total_steps=100
Fixed init: False
Device: cpu


######################################################################
# SETTING: K=4
######################################################################

  --- d=100, p=505, n_train=252, n/p=0.499, K=4 ---
    Convergence (rep 1, full-data fit, n=252):
      Epoch 2/7, Loss: 0.451606
      Epoch 4/7, Loss: 0.193440
      Epoch 6/7, Loss: 0.115525
    Train loss (final, rep 1): 0.095342
    Test MSE (mean ± std): 0.510875 ± 0.065267
    KCV (mean ± std): 0.605386 ± 0.079104
    Estimation error (Test - KCV): mean=-0.094511, std=0.103952
    Mean |KCV - Test MSE|: 0.106780

  --- d=200, p=1005, n_train=502, n/p=0.500, K=4 ---
    Convergence (rep 1, full-data fit, n=502):
      Epoch 1/4, Loss: 0.529742
      Epoch 2/4, Loss: 0.212435
      Epoch 3/4, Loss: 0.131380
      Epoch 4/4, Loss: 0.087733
    Train loss (final, re

In [ ]:
import os
print("Working dir:", os.getcwd())
print("Contents of cwd:", os.listdir())
print("Contents of /content:", os.listdir('/content'))

Working dir: /content
Contents of cwd: ['.config', 'sample_data']
Contents of /content: ['.config', 'sample_data']


In [ ]:
# ==== FIGURE CELL: run after the simulation cell ====
# Requires all_rate_points, which the sim cell now fills for every K setting.
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as sps

for rule_name, pts in all_rate_points.items():        # one figure per K setting
    ps = np.array([q[0] for q in pts], dtype=float)
    ad = np.array([q[1] for q in pts], dtype=float)
    keep = ad > 0
    ps, ad = ps[keep], ad[keep]
    if len(ps) < 3:
        continue

    y = 2.0 * np.log(ad)
    reg = sps.linregress(np.log(ps), y)
    t_crit = sps.t.ppf(0.975, len(ps) - 2)
    ci_lo = reg.slope - t_crit * reg.stderr
    ci_hi = reg.slope + t_crit * reg.stderr

    unique_ps = np.sort(np.unique(ps))
    positions = np.log(unique_ps)
    width = 0.5 * np.min(np.diff(positions)) if len(positions) > 1 else 0.4

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.boxplot([y[ps == u] for u in unique_ps], positions=positions, widths=width)
    x_line = np.linspace(positions[0], positions[-1], 100)
    ax.plot(x_line, reg.intercept + reg.slope * x_line, linewidth=2.5,
            label=f"Slope: {reg.slope:.4f}, 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    ax.set_xticks(positions)
    ax.set_xticklabels([str(int(u)) for u in unique_ps])
    ax.set_xlabel("p")
    ax.set_ylabel("2 log absolute difference")
    ax.set_title(f"Log of Absolute Difference Between CV MSE and OOS MSE, {rule_name}")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

NameError: name 'all_rate_points' is not defined